# Mimir no-LGTM assessment

Assesses Mimir **without using the LGTM stack as an instrument** -- kube API state,
events, PVCs, and direct HTTP `/ready` probes only. This is the realistic tool for
the not-uncommon case where Mimir is unhappy and therefore can't answer questions
about itself (metrics, dashboards, and Loki-via-the-same-cluster may all be lying
or dark). The gateway query smoke test at the end touches Mimir, but as the system
under test, never as an instrument.

Normal-times status: [mimir-health.ipynb](mimir-health.ipynb); capacity:
[mimir-usage.ipynb](mimir-usage.ipynb). Design: [tiles#644](https://github.com/symmatree/tiles/issues/644).

Run from `notebooks/`: `jupyter nbconvert --to notebook --execute --inplace mimir-nolgtm.ipynb`

In [1]:
namespace = "mimir"
gateway_url = "http://mimir-gateway.mimir.svc"   # smoke test target only
tenant = "tiles"                                 # X-Scope-OrgID for the smoke test
capture_file = ""      # replay a prior raw capture JSON instead of querying live
output_dir = ""        # if set: write raw capture + agent stats there
debug = False

In [2]:
import json
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import nb_capture as nbc

NOTEBOOK = 'mimir-nolgtm'
cap = nbc.Capture(replay_file=capture_file, namespace=namespace,
                  gateway_url=gateway_url, tenant=tenant)
namespace, tenant = cap.meta['namespace'], cap.meta['tenant']
kube = nbc.Kube(cap)
print(f"namespace: {namespace}  "
      f"{'REPLAY of ' + cap.run_at if cap.replay else 'live'}")

namespace: mimir  live


In [3]:
# --- ASSUMPTIONS: only that the kube API answers (Kube.get raises otherwise).
# Deliberately nothing else -- every other system here is under test, not trusted.
ns = kube.get('namespace', 'namespace', namespace)
print(f"kube API: ok; namespace {namespace} is {ns['status']['phase']}")

kube API: ok; namespace mimir is Active


In [4]:
# --- PODS: phase, readiness, restarts, last terminations (kube API view) ---
pods_raw = kube.get('pods', 'pods', '-n', namespace)
rows = []
for p in pods_raw['items']:
    cs = p['status'].get('containerStatuses', [])
    term = next(((c['lastState'].get('terminated') or {}) for c in cs
                 if c['lastState'].get('terminated')), {})
    rows.append({'pod': p['metadata']['name'],
                 'phase': p['status'].get('phase', '?'),
                 'ready': f"{sum(c['ready'] for c in cs)}/{len(cs)}",
                 'restarts': sum(c['restartCount'] for c in cs),
                 'node': p['spec'].get('nodeName', ''),
                 'last_term': f"{term.get('reason', '')} {(term.get('finishedAt') or '')[:16]}".strip()})
pods_df = pd.DataFrame(rows).set_index('pod').sort_index()
ready_n = pods_df['ready'].str.split('/', expand=True)
not_ready = sorted(pods_df.index[(pods_df['phase'] != 'Running')
                                 | (ready_n[0] != ready_n[1])])
print(pods_df.to_string())
print(f"not ready: {not_ready or 'none'}")

                                               phase ready  restarts        node               last_term
pod                                                                                                     
mimir-alertmanager-0                         Running   2/2         0  tiles-wk-1                        
mimir-compactor-0                            Running   1/1         0  tiles-wk-2                        
mimir-distributor-7ccf8d9fdf-lc5x7           Running   1/1         3  tiles-wk-2  Error 2026-07-24T19:02
mimir-gateway-d47758865-mntfq              Succeeded   0/1         0  tiles-wk-1                        
mimir-gateway-d47758865-tqlxk                Running   1/1         0  tiles-wk-3                        
mimir-ingester-0                             Running   1/1         9  tiles-wk-1  Error 2026-07-25T16:33
mimir-ingester-1                             Running   1/1         2  tiles-wk-2  Error 2026-07-24T19:02
mimir-kafka-0                                Running   

In [5]:
# --- EVENTS: Warning events in the namespace (whatever etcd still holds) ---
ev = kube.get('events', 'events', '-n', namespace)

def ev_time(e):
    return e.get('lastTimestamp') or e.get('eventTime') or e['metadata']['creationTimestamp']

warn = sorted((e for e in ev['items'] if e.get('type') == 'Warning'),
              key=ev_time, reverse=True)
warn_reasons = dict(Counter(e.get('reason', '?') for e in warn))
print(f"{len(ev['items'])} events retained, {len(warn)} warnings; by reason: {warn_reasons or 'none'}")
for e in warn[:15]:
    print(f"  {ev_time(e)[:19]} x{e.get('count') or 1:<3} {e.get('reason', '?'):<18} "
          f"{e['involvedObject'].get('name', '')[:38]:<38} {e.get('message', '')[:100]}")

3 events retained, 2 warnings; by reason: {'NodeNotReady': 1, 'Unhealthy': 1}
  2026-07-29T18:11:33 x2   NodeNotReady       mimir-gateway-d47758865-tqlxk          Node is not ready
  2026-07-29T18:10:47 x1   Unhealthy          mimir-ruler-748b9f9c9c-zq4d7           Readiness probe failed: Get "http://10.0.149.253:8080/ready": context deadline exceeded (Client.Time


In [6]:
# --- DIRECT /ready PROBES: each component service, no metrics involved ---
svcs = kube.get('services', 'services', '-n', namespace)
targets = sorted(s['metadata']['name'] for s in svcs['items']
                 if not s['metadata']['name'].endswith('-headless')
                 and any(p.get('name') == 'http-metrics' and p['port'] == 8080
                         for p in s['spec']['ports']))
probe = {}
for nm in targets:
    rec = cap.http(f'ready: {nm}', f'http://{nm}.{namespace}.svc:8080/ready', timeout=5)
    body = (rec.get('text') or json.dumps(rec.get('json', ''))).strip()[:60]
    probe[nm] = rec.get('error') or f"{rec.get('status')} {body}"
    print(f"  {nm:<28} {probe[nm]}")
ready_failed = sorted(nm for nm, r in probe.items() if not r.startswith('200'))
print(f"failed probes: {ready_failed or 'none'}")

  mimir-alertmanager           200 ready
  mimir-compactor              200 ready
  mimir-distributor            200 ready
  mimir-ingester               200 ready
  mimir-overrides-exporter     200 ready
  mimir-querier                200 ready
  mimir-query-frontend         200 ready
  mimir-query-scheduler        200 ready
  mimir-ruler                  200 ready
  mimir-store-gateway          200 ready
failed probes: none


In [7]:
# --- PVCs ---
pvcs = kube.get('pvcs', 'pvc', '-n', namespace)
pvc_phase = {p['metadata']['name']: p['status']['phase'] for p in pvcs['items']}
pvc_not_bound = sorted(n for n, ph in pvc_phase.items() if ph != 'Bound')
print(f"{len(pvc_phase)} PVCs; not bound: {pvc_not_bound or 'none'}")

7 PVCs; not bound: none


In [8]:
# --- SMOKE TEST (system under test, not an instrument): does the query path answer? ---
smoke = {}
for nm, path, params in [('buildinfo', '/prometheus/api/v1/status/buildinfo', {}),
                         ('query', '/prometheus/api/v1/query', {'query': 'vector(1)'})]:
    rec = cap.http(f'smoke: {nm}', f"{cap.meta['gateway_url']}{path}", params=params,
                   headers={'X-Scope-OrgID': tenant}, timeout=10)
    smoke[nm] = rec.get('error') or rec.get('status')
    print(f"  {nm:<10} {smoke[nm]}")
query_path_ok = smoke.get('query') == 200
print(f"query path answers: {query_path_ok}")

  buildinfo  200
  query      200
query path answers: True


In [9]:
# --- SUMMARY ---
findings = []
if not_ready:
    findings.append(f"pods not ready: {not_ready}")
if ready_failed:
    findings.append(f"/ready probes failing: {ready_failed}")
if pvc_not_bound:
    findings.append(f"PVCs not bound: {pvc_not_bound}")
if not query_path_ok:
    findings.append(f"gateway query path NOT answering: {smoke}")
bad_reasons = {r: n for r, n in warn_reasons.items()
               if r in ('OOMKilling', 'BackOff', 'CrashLoopBackOff', 'FailedScheduling',
                        'FailedMount', 'Unhealthy', 'Evicted')}
if bad_reasons:
    findings.append(f"warning events of concern: {bad_reasons}")

print("=" * 60)
print("MIMIR NO-LGTM SUMMARY")
print("=" * 60)
print(f"run:      {cap.run_at}  ({cap.mode})")
print(f"scope:    namespace {namespace}, kube API + direct HTTP only")
print()
if findings:
    print("Findings:")
    for f in findings:
        print(f"  ! {f}")
else:
    print(f"Findings: none -- {len(pods_df)} pods running/ready, "
          f"{len(probe)} /ready probes green, PVCs bound, query path answers")

agent_stats = {
    'notebook': f'{NOTEBOOK}.ipynb',
    'run_at': cap.run_at,
    'mode': cap.mode,
    'namespace': namespace,
    'findings': findings,
    'pods': {'total': len(pods_df), 'not_ready': not_ready,
             'restarts_by_pod': {p: int(n) for p, n in pods_df['restarts'].items() if n > 0},
             'last_term_by_pod': {p: t for p, t in pods_df['last_term'].items() if t}},
    'events': {'warning_count': len(warn), 'warning_by_reason': warn_reasons},
    'ready_probes': probe,
    'pvc': pvc_phase,
    'smoke': {k: str(v) for k, v in smoke.items()},
}
print()
print(json.dumps(agent_stats, indent=1))

if output_dir:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cap.save(out / f'{NOTEBOOK}.capture.json')
    (out / f'{NOTEBOOK}.stats.json').write_text(json.dumps(agent_stats, indent=2))
    print(f"wrote {out / f'{NOTEBOOK}.capture.json'} and {out / f'{NOTEBOOK}.stats.json'}")

MIMIR NO-LGTM SUMMARY
run:      2026-07-29T18:36:20.397878+00:00  (live)
scope:    namespace mimir, kube API + direct HTTP only

Findings:
  ! pods not ready: ['mimir-gateway-d47758865-mntfq', 'mimir-overrides-exporter-56bbcb4b97-729st', 'mimir-query-scheduler-78b7cd94b4-98vdg', 'mimir-query-scheduler-78b7cd94b4-s7srq', 'mimir-ruler-748b9f9c9c-gzxbw']
  ! warning events of concern: {'Unhealthy': 1}

{
 "notebook": "mimir-nolgtm.ipynb",
 "run_at": "2026-07-29T18:36:20.397878+00:00",
 "mode": "live",
 "namespace": "mimir",
 "findings": [
  "pods not ready: ['mimir-gateway-d47758865-mntfq', 'mimir-overrides-exporter-56bbcb4b97-729st', 'mimir-query-scheduler-78b7cd94b4-98vdg', 'mimir-query-scheduler-78b7cd94b4-s7srq', 'mimir-ruler-748b9f9c9c-gzxbw']",
  "warning events of concern: {'Unhealthy': 1}"
 ],
 "pods": {
  "total": 20,
  "not_ready": [
   "mimir-gateway-d47758865-mntfq",
   "mimir-overrides-exporter-56bbcb4b97-729st",
   "mimir-query-scheduler-78b7cd94b4-98vdg",
   "mimir-query-sc